In [7]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader


from uji_utils import make_features
from uji_nn import MLPMultiTask, MLPJoint, TrainConfig, train_multitask_classifier, train_joint_classifier
from uji_boost import train_xgb_joint

In [8]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

device: cpu


In [9]:
DATA_DIR = Path('../../data/UjiIndoorLoc') # fragile, later redo this
train = pd.read_csv(DATA_DIR / 'TrainingData.csv')
val = pd.read_csv(DATA_DIR / 'ValidationData.csv')

wap_cols = [c for c in train.columns if c.startswith('WAP')]
print('train shape:', train.shape, 'val shape:', val.shape)
print('n_wap:', len(wap_cols))
print('WAP min/max in train:', train[wap_cols].to_numpy().min(), train[wap_cols].to_numpy().max())
print('fraction no-signal (==100):', (train[wap_cols].to_numpy() == 100).mean())

train shape: (19937, 529) val shape: (1111, 529)
n_wap: 520
WAP min/max in train: -104 100
fraction no-signal (==100): 0.965394550526466


In [10]:
X_train = make_features(train, wap_cols).copy()
X_val = make_features(val, wap_cols).copy()

yb_train = train['BUILDINGID'].to_numpy(dtype=np.int64, copy=True)
yf_train = train['FLOOR'].to_numpy(dtype=np.int64, copy=True)
yb_val = val['BUILDINGID'].to_numpy(dtype=np.int64, copy=True)
yf_val = val['FLOOR'].to_numpy(dtype=np.int64, copy=True)

print('X_train:', X_train.shape, 'X_val:', X_val.shape)
print('building classes:', np.unique(yb_train))
print('floor classes:', np.unique(yf_train))

X_train: (19937, 1040) X_val: (1111, 1040)
building classes: [0 1 2]
floor classes: [0 1 2 3 4]


## XGBoost (joint building+floor) on shared features

In [11]:
from uji_utils import build_joint_labels

# Shared joint BUILDINGID_FLOOR label for both XGBoost and single-task NN
y_train_joint, y_val_joint, le = build_joint_labels(train, val)

print('joint classes:', len(le.classes_))

joint classes: 13


In [12]:
xgb_result = train_xgb_joint(X_train, X_val, train, val)

print(f"[XGBoost] Joint (building+floor) accuracy: {xgb_result.joint_acc:.4f}")
print(f"[XGBoost] Building accuracy:              {xgb_result.b_acc:.4f}")
print(f"[XGBoost] Floor accuracy:                 {xgb_result.f_acc:.4f}")

[XGBoost] Joint (building+floor) accuracy: 0.8929
[XGBoost] Building accuracy:              0.9802
[XGBoost] Floor accuracy:                 0.8956


## Base NN (MLP) on the same features

In [13]:
batch_size = 256
train_ds = TensorDataset(
    torch.from_numpy(X_train),
    torch.from_numpy(yb_train),
    torch.from_numpy(yf_train),
)
val_ds = TensorDataset(
    torch.from_numpy(X_val),
    torch.from_numpy(yb_val),
    torch.from_numpy(yf_val),
)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=1024, shuffle=False, num_workers=0)
print('n_train:', len(train_ds), 'n_val:', len(val_ds))

n_train: 19937 n_val: 1111


In [14]:
model = MLPMultiTask(in_dim=X_train.shape[1], hidden=(1024, 512, 256), p_drop=0.15).to(device)

cfg_mt = TrainConfig(lr=2e-3, weight_decay=1e-4, floor_loss_weight=1.1, max_epochs=200, patience=20, print_every=5)
mt_metrics, mt_state = train_multitask_classifier(model, train_loader, val_loader, device, cfg_mt)

print('--- NN multi-task best ---')
print('best_epoch:', int(mt_metrics['best_epoch']))
print(f"[NN MT] Building accuracy: {mt_metrics['b_acc']:.4f}")
print(f"[NN MT] Floor accuracy:    {mt_metrics['f_acc']:.4f}")
print(f"[NN MT] Joint accuracy:    {mt_metrics['joint']:.4f}")

epoch=01 loss=0.2521 b_acc=0.9991 f_acc=0.8749 joint=0.8749 lr=0.002000
epoch=05 loss=0.0253 b_acc=0.9991 f_acc=0.8911 joint=0.8911 lr=0.002000
epoch=10 loss=0.0117 b_acc=0.9982 f_acc=0.8974 joint=0.8974 lr=0.001000
epoch=15 loss=0.0089 b_acc=0.9982 f_acc=0.8911 joint=0.8911 lr=0.000500
epoch=20 loss=0.0076 b_acc=0.9982 f_acc=0.8938 joint=0.8938 lr=0.000250
--- NN multi-task best ---
best_epoch: 4
[NN MT] Building accuracy: 0.9982
[NN MT] Floor accuracy:    0.9028
[NN MT] Joint accuracy:    0.9028


## Single-task NN: joint building+floor label

Now we add a **single-head MLP** that predicts the combined `BUILDINGID_FLOOR` class (same 13-way label used for XGBoost). This lets us compare:
- XGBoost joint classifier
- Multi-task NN (separate building/floor heads)
- **Single-task NN** on the exact same features.

In [15]:
# Datasets/loaders for joint label
joint_train_ds = TensorDataset(
    torch.from_numpy(X_train),
    torch.from_numpy(y_train_joint),
)
joint_val_ds = TensorDataset(
    torch.from_numpy(X_val),
    torch.from_numpy(y_val_joint),
)

joint_train_loader = DataLoader(joint_train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
joint_val_loader = DataLoader(joint_val_ds, batch_size=1024, shuffle=False, num_workers=0)

print('n_train (joint):', len(joint_train_ds), 'n_val (joint):', len(joint_val_ds))

n_train (joint): 19937 n_val (joint): 1111


In [16]:
# Joint NN trained on combined BUILDINGID_FLOOR label
joint_model = MLPJoint(in_dim=X_train.shape[1], n_classes=len(le.classes_), hidden=(1024, 512, 256), p_drop=0.15).to(device)

cfg_joint = TrainConfig(lr=2e-3, weight_decay=1e-4, max_epochs=200, patience=20, print_every=5)
joint_metrics, joint_state = train_joint_classifier(joint_model, joint_train_loader, joint_val_loader, device, le.classes_, cfg_joint)

print('--- Joint NN best ---')
print('best_epoch:', int(joint_metrics['best_epoch']))
print(f"[Joint NN] Joint (building+floor) accuracy: {joint_metrics['joint']:.4f}")
print(f"[Joint NN] Building accuracy:          {joint_metrics['b_acc']:.4f}")
print(f"[Joint NN] Floor accuracy:             {joint_metrics['f_acc']:.4f}")

epoch=01 loss=0.2375 joint=0.8308 b_acc=0.9973 f_acc=0.8308 lr=0.002000
epoch=05 loss=0.0261 joint=0.8929 b_acc=0.9991 f_acc=0.8929 lr=0.002000
epoch=10 loss=0.0167 joint=0.9073 b_acc=0.9955 f_acc=0.9073 lr=0.002000
epoch=15 loss=0.0140 joint=0.9028 b_acc=0.9964 f_acc=0.9028 lr=0.001000
epoch=20 loss=0.0053 joint=0.9046 b_acc=0.9964 f_acc=0.9046 lr=0.000500
epoch=25 loss=0.0046 joint=0.9082 b_acc=0.9955 f_acc=0.9082 lr=0.000500
epoch=30 loss=0.0042 joint=0.9046 b_acc=0.9937 f_acc=0.9046 lr=0.000250
epoch=35 loss=0.0039 joint=0.9037 b_acc=0.9955 f_acc=0.9037 lr=0.000125
epoch=40 loss=0.0039 joint=0.9046 b_acc=0.9937 f_acc=0.9046 lr=0.000063
epoch=45 loss=0.0038 joint=0.9028 b_acc=0.9937 f_acc=0.9028 lr=0.000031
--- Joint NN best ---
best_epoch: 25
[Joint NN] Joint (building+floor) accuracy: 0.9082
[Joint NN] Building accuracy:          0.9955
[Joint NN] Floor accuracy:             0.9082
